# <div align="center"> Statistical Learning </div>
# <div align="center"> Machine Learning and Econometrics </div>
## <div align="center"> ECO 4971/6971 </div>
#### <div align="center">Class 10 — Model Selection (Ridge, Lasso): Exercise Solutions</div>
<div align="center"> Jonathan Holmes (he/him)</div>

These exercises use the `Credit.csv` dataset.

# Exercise 1 — Best Subset Selection

**Q1. Which single variable does the best job predicting `Balance`?**

---

**Answer.**

Best-subset selection with one predictor picks the variable with the highest $R^2$ (equivalently, lowest RSS) when regressed individually on `Balance`. In `Credit.csv`, the variable with the strongest simple linear relationship with credit card balance is **`Rating`** (credit rating), which has a correlation of roughly 0.86 with `Balance`.

`Income` is the second strongest individual predictor. The remaining variables (Limit, Age, Cards, Education, etc.) are weaker individually.

---

**Q2. How many possible subsets are there if you have $k$ predictors?**

---

**Answer.**

For each predictor we make a binary choice: include it or not. With $k$ predictors there are $2^k$ total subsets (including the null model with no predictors). Excluding the null model, there are $2^k - 1$ non-empty subsets.

| Predictors ($k$) | Non-empty subsets ($2^k - 1$) |
|:---:|:---:|
| 1 | 1 |
| 2 | 3 |
| 3 | 7 |
| 4 | 15 |
| 5 | 31 |

The number grows exponentially. The `Credit` dataset has 10 predictors, giving $2^{10} - 1 = 1{,}023$ subsets — feasible, but with $p = 30$ predictors we already have over $10^9$ subsets, making exhaustive search computationally infeasible. This motivates **stepwise** and **shrinkage** methods.

# Exercise 2 — Ridge Regression

**Q3. Why do we call Ridge regression a "shrinkage" estimator?**

---

**Answer.**

Ridge regression minimises a penalised criterion:

$$\text{RSS} + \lambda \sum_{j=1}^{p} \beta_j^2 = \sum_{i=1}^n (y_i - \hat{y}_i)^2 + \lambda \|\boldsymbol{\beta}\|_2^2$$

The $\ell_2$ penalty $\lambda\sum_j \beta_j^2$ penalises large coefficients. As $\lambda$ increases, the optimal $\hat{\boldsymbol{\beta}}^{\text{Ridge}}$ is pushed toward zero — the coefficients are **shrunk** relative to the OLS estimates $\hat{\boldsymbol{\beta}}^{\text{OLS}}$.

The closed-form solution makes this explicit:

$$\hat{\boldsymbol{\beta}}^{\text{Ridge}} = (\mathbf{X}^\top\mathbf{X} + \lambda \mathbf{I})^{-1}\mathbf{X}^\top\mathbf{y}$$

When $\lambda = 0$ this reduces to OLS; as $\lambda \to \infty$ all coefficients shrink to 0. The name "shrinkage" refers specifically to this pulling of estimates toward zero.

---

**Q4. Suppose we predict farm harvest with:**

$$\text{TotalHarvest} = \beta_0 + \beta_1\,\text{AverageTemperature} + \beta_2\,\text{AverageRainfall} + u$$

**(a) If I run OLS with temperature in °C and then in °K, what changes?**

---

**Answer (a).**

Kelvin = Celsius + 273.15, a **linear (affine) transformation**. OLS is **equivariant** under linear transformations of the predictors: the coefficient on Temperature simply rescales to keep the fitted values identical.

* $\hat{\beta}_1^{(K)} = \hat{\beta}_1^{(C)}$ — since the *slope* does not change when we add a constant (the constant is absorbed into the intercept).
* The **intercept changes**: $\hat{\beta}_0^{(K)} = \hat{\beta}_0^{(C)} - 273.15\,\hat{\beta}_1$.
* All other quantities — fitted values, residuals, $R^2$, $p$-values — are **identical**.

*(For a unit rescaling like °C → °F the slope would change by the conversion factor, but the fitted values and $R^2$ remain identical.)*

---

**(b) If I run Ridge regression with temperature in °C and then °K, what changes?**

---

**Answer (b).**

Ridge is **not** equivariant under rescaling. The $\ell_2$ penalty $\lambda(\beta_1^2 + \beta_2^2)$ penalises the raw coefficient values. If we change temperature from °C to °K (adding 273.15), the scale of $X_1$ changes, so the ridge penalty treats the same physical information differently depending on units.

Concretely, the same $\lambda$ will impose a *different* amount of shrinkage on $\hat{\beta}_1$ depending on whether temperature is in °C or °K. The fitted values and test MSE will differ.

**This is why we always standardise predictors before applying Ridge (or Lasso):** standardisation ensures all predictors enter the penalty on equal footing, making the shrinkage unit-free.

# Exercise 3 — Lasso

**Q5. What is the difference between Lasso and Ridge regression?**

---

**Answer.**

Both methods add a penalty to OLS to shrink coefficients, but they differ in the **type of penalty**:

| | **Ridge** | **Lasso** |
|---|---|---|
| Penalty | $\lambda \sum_j \beta_j^2$ ($\ell_2$ norm) | $\lambda \sum_j |\beta_j|$ ($\ell_1$ norm) |
| Coefficients | Shrunk toward zero, never exactly zero | Can be set **exactly to zero** |
| Variable selection | No (all variables retained) | **Yes** (sparse solutions) |
| Geometry | Circular constraint region | Diamond constraint region — corners at axes produce zeros |
| Best when | Many small effects (dense signal) | Few large effects, many irrelevant predictors (sparse signal) |

The key practical difference: **Lasso performs automatic variable selection** by setting irrelevant coefficients to exactly zero, producing an interpretable sparse model. Ridge keeps all variables but dampens their influence.

---

**Q6. How would you decide among Lasso, Ridge, and OLS in practice?**

---

**Answer.**

Use **cross-validation** to compare prediction error across the three approaches (and across values of $\lambda$ for Ridge and Lasso). The practical decision logic:

1. **When to prefer OLS:** Few predictors relative to observations, no reason to believe many are irrelevant. OLS is unbiased and interpretable; regularisation would only add unnecessary bias.

2. **When to prefer Ridge:** Many predictors that may all contribute a little (dense signal), or multicollinearity among predictors. Ridge stabilises estimates by shrinking correlated coefficients together.

3. **When to prefer Lasso:** Many predictors but only a few are truly relevant (sparse signal). Lasso's variable selection yields an interpretable model and can improve prediction when many predictors are noise.

4. **Elastic Net** (a blend of Ridge and Lasso penalties) is useful when predictors are correlated and sparsity is still desired — Lasso tends to arbitrarily pick one among highly correlated predictors, while Elastic Net can retain groups.

In all cases, **tune $\lambda$ via cross-validation** on a training set and report final performance on a held-out test set.